# Tarefa 1 — Problema e Coleta Bruta (sem rótulo)

**Projeto:** Preditor de degradação de rede com RTT normalizado  
**Equipe:** ICMPMASTER

## Objetivo
Definir o problema, conferir a coleta pública do RIPE Atlas, preservar os dados brutos e separar os períodos A e B sem criar rótulos.

> **Limite desta tarefa:** não calcular baseline, `z_robusto`, classe `OK/RISCO/FALHA`, recortes de treino/validação/teste nem treinar árvore. RTT absoluto não define falha.

Execute este notebook a partir da raiz do repositório.


## 0. Contrato e estrutura esperada

Entradas principais:
- `config/coleta_config.json`: parâmetros de coleta e períodos;
- `raw/api_ripe/`: JSONs originais da API;
- `requirements.txt`: dependências.

Saídas de trabalho:
- `data/interim/periodo_a_bruto.csv`;
- `data/interim/periodo_b_bruto.csv`;
- relatório de qualidade **sem classe**.

Os JSONs em `raw/` devem permanecer intactos. Se a coleta ainda não foi executada, este notebook não inventa dados nem substitui a coleta real.


In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np

RAIZ = Path.cwd()
CONFIG_PATH = RAIZ / "config" / "coleta_config.json"
RAW_DIR = RAIZ / "raw" / "api_ripe"
INTERIM_DIR = RAIZ / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz:", RAIZ)
print("Configuração encontrada:", CONFIG_PATH.exists())
print("Pasta de JSONs encontrada:", RAW_DIR.exists())

if CONFIG_PATH.exists():
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    print("Chaves da configuração:", list(config.keys()))
else:
    print("Configure/coloque config/coleta_config.json na raiz antes de continuar.")


## 1. Definição do problema

- **Evento futuro a classificar:** degradação do fluxo em relação ao comportamento normal do próprio fluxo.
- **Unidade de análise:** `fluxo_id = probe_id | dst_addr`.
- **Cada linha bruta:** uma medição ICMP com timestamp, ainda sem classe.
- **Fonte:** medições públicas de ping IPv4 já existentes no RIPE Atlas, consultadas por `GET`.
- **Proibições:** não definir falha por RTT absoluto, país, continente ou nome da rota.

A rota serve apenas para documentar diversidade de caminhos curtos e longos; não vira classe nem variável de entrada do modelo.


## 2. Inventário e preservação dos dados brutos

A célula seguinte inventaria arquivos JSON e calcula hashes antes de qualquer processamento. Isso ajuda a demonstrar que os originais não foram modificados.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

json_paths = sorted(RAW_DIR.rglob("*.json")) if RAW_DIR.exists() else []
inventario = []
for p in json_paths:
    inventario.append({
        "arquivo": str(p.relative_to(RAIZ)),
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p)
    })

inventario_df = pd.DataFrame(inventario)
print("Arquivos JSON encontrados:", len(inventario_df))
display(inventario_df.head(20))


## 3. Leitura segura dos JSONs

Como os formatos de resposta podem variar, esta etapa apenas tenta identificar registros de medição. Ela não altera os arquivos originais. Se o formato do JSON não corresponder ao esperado, revise a estrutura antes de continuar.


In [ ]:
def carregar_registros_json(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        # Algumas exportações podem envolver a lista em uma chave.
        for chave in ("results", "data", "measurements"):
            if isinstance(obj.get(chave), list):
                return obj[chave]
    return []

registros = []
arquivos_sem_registros = []
for p in json_paths:
    linhas = carregar_registros_json(p)
    if not linhas:
        arquivos_sem_registros.append(str(p.relative_to(RAIZ)))
    for linha in linhas:
        if isinstance(linha, dict):
            registros.append(linha)

print("Registros JSON identificados:", len(registros))
print("Arquivos sem lista de registros reconhecida:", len(arquivos_sem_registros))
if registros:
    print("Campos de exemplo:", sorted(registros[0].keys()))
else:
    print("Nenhum registro foi carregado. Confira o formato dos JSONs em raw/api_ripe/.")


## 4. Padronização mínima para auditoria

Esta etapa procura nomes comuns de campos da API. A tabela de trabalho mantém RTT ausente como ausente — nunca transforma ausência em `0`. Confira o mapeamento antes de usar as saídas.


In [ ]:
def primeiro_valor(reg, *chaves):
    for chave in chaves:
        if chave in reg and reg[chave] is not None:
            return reg[chave]
    return np.nan

linhas = []
for r in registros:
    timestamp = primeiro_valor(r, "timestamp", "time")
    try:
        timestamp = pd.to_datetime(timestamp, unit="s", utc=True) if isinstance(timestamp, (int, float)) else pd.to_datetime(timestamp, utc=True)
    except Exception:
        timestamp = pd.NaT
    probe_id = primeiro_valor(r, "prb_id", "probe_id")
    dst_addr = primeiro_valor(r, "dst_addr", "dst")
    sent = pd.to_numeric(primeiro_valor(r, "sent"), errors="coerce")
    rcvd = pd.to_numeric(primeiro_valor(r, "rcvd", "received"), errors="coerce")
    latencia = pd.to_numeric(primeiro_valor(r, "avg", "latencia_ms", "rtt"), errors="coerce")
    if pd.isna(sent) and pd.notna(rcvd):
        sent = 3
    perda = ((sent - rcvd) / sent * 100) if pd.notna(sent) and sent > 0 and pd.notna(rcvd) else np.nan
    linhas.append({
        "timestamp": timestamp,
        "measurement_id": primeiro_valor(r, "msm_id", "measurement_id"),
        "probe_id": probe_id,
        "dst_addr": dst_addr,
        "fluxo_id": f"{probe_id}|{dst_addr}" if pd.notna(probe_id) and pd.notna(dst_addr) else np.nan,
        "latencia_ms": latencia,
        "sent": sent,
        "rcvd": rcvd,
        "perda_pct": perda,
        "timeout_atual": int(pd.isna(latencia) or (pd.notna(perda) and perda >= 100))
    })

bruto = pd.DataFrame(linhas)
print("Linhas padronizadas:", len(bruto))
display(bruto.head())


## 5. Configuração dos períodos A e B

A configuração deve definir os limites de A e B. A célula tenta ler a estrutura `periodos.A.inicio/fim` e `periodos.B.inicio/fim`. Se as chaves tiverem outro formato no arquivo real, ajuste o mapeamento explicitamente; não escolha datas silenciosamente.


In [ ]:
def obter_periodo(config, nome):
    bloco = config.get("periodos", {}).get(nome, {})
    inicio = bloco.get("inicio")
    fim = bloco.get("fim")
    if not inicio or not fim:
        raise KeyError(f"Configuração não contém periodos.{nome}.inicio/fim")
    return pd.to_datetime(inicio, utc=True), pd.to_datetime(fim, utc=True)

if CONFIG_PATH.exists() and not bruto.empty:
    A0, A1 = obter_periodo(config, "A")
    B0, B1 = obter_periodo(config, "B")
    print("A:", A0, "até", A1, "(fim exclusivo)")
    print("B:", B0, "até", B1, "(fim exclusivo)")
    print("A e B contíguos:", A1 == B0)

    bruto["timestamp"] = pd.to_datetime(bruto["timestamp"], utc=True, errors="coerce")
    periodo_a = bruto.loc[(bruto["timestamp"] >= A0) & (bruto["timestamp"] < A1)].copy()
    periodo_b = bruto.loc[(bruto["timestamp"] >= B0) & (bruto["timestamp"] < B1)].copy()
else:
    periodo_a = pd.DataFrame()
    periodo_b = pd.DataFrame()
    print("Períodos não separados: verifique configuração e presença dos registros.")


## 6. Relatório de qualidade — ainda sem classe

O relatório resume volume, cobertura temporal, valores ausentes, duplicatas e timeouts. Não interpreta RTT alto como falha.


In [ ]:
if not bruto.empty:
    duplicatas = bruto.duplicated(subset=["timestamp", "fluxo_id"], keep=False).sum()
    relatorio = {
        "n_registros": len(bruto),
        "n_fluxos": bruto["fluxo_id"].nunique(dropna=True),
        "n_timestamps_ausentes": bruto["timestamp"].isna().sum(),
        "n_fluxo_id_ausente": bruto["fluxo_id"].isna().sum(),
        "n_rtt_ausente": bruto["latencia_ms"].isna().sum(),
        "n_duplicatas_timestamp_fluxo": int(duplicatas),
        "n_timeouts": int(bruto["timeout_atual"].sum()),
        "rtt_min_ms": bruto["latencia_ms"].min(),
        "rtt_mediana_ms": bruto["latencia_ms"].median(),
        "rtt_max_ms": bruto["latencia_ms"].max(),
        "perda_mediana_pct": bruto["perda_pct"].median(),
        "n_periodo_a": len(periodo_a),
        "n_periodo_b": len(periodo_b),
    }
    display(pd.DataFrame([relatorio]).T.rename(columns={0:"valor"}))
    print("Coluna classe criada?", "classe" in bruto.columns)
else:
    print("Relatório não calculado porque não há registros reconhecidos.")


## 7. Validações e exportação

Antes de exportar, confirmamos que não existe coluna de classe e que os períodos não se sobrepõem. Os CSVs são arquivos derivados; os JSONs originais permanecem intocados.


In [ ]:
if not bruto.empty and not periodo_a.empty and not periodo_b.empty:
    assert "classe" not in bruto.columns
    assert not (set(periodo_a.index) & set(periodo_b.index))
    assert not ((periodo_a["timestamp"] >= B0) & (periodo_a["timestamp"] < B1)).any()
    assert not ((periodo_b["timestamp"] >= A0) & (periodo_b["timestamp"] < A1)).any()

    periodo_a.to_csv(INTERIM_DIR / "periodo_a_bruto.csv", index=False, encoding="utf-8")
    periodo_b.to_csv(INTERIM_DIR / "periodo_b_bruto.csv", index=False, encoding="utf-8")
    bruto.to_csv(INTERIM_DIR / "medicoes_brutas_auditadas.csv", index=False, encoding="utf-8")
    print("CSV derivados gravados em:", INTERIM_DIR)
else:
    print("Exportação não realizada: confira os dados e os períodos antes de gerar saídas.")


## 8. Scrum, diário de bordo e evidências

**Histórias de usuário**
1. Coletar fluxos diversos, incluindo caminhos curtos e longos.
2. Preservar o bruto com timeouts e sem converter RTT ausente em zero.
3. Separar períodos A e B sem sobreposição e sem rótulos.

**Link do board:** preencher com o link real da equipe.

| Integrante | O que fiz nesta tarefa | Dificuldades | O que pretendo manter/ajustar |
|---|---|---|---|
| | | | |

**Evidências**
- RFC:
- Dicionário v0.1:
- Commits:
- Configuração da coleta:
- Arquivos brutos preservados:


## 9. Critério de conclusão

- [ ] Fluxos curtos e longos documentados.
- [ ] JSON original preservado e hashes registrados.
- [ ] Timeout preservado; RTT ausente não foi convertido em zero.
- [ ] Períodos A e B sem sobreposição.
- [ ] Nenhuma classe, baseline ou métrica relativa foi criada.
- [ ] Link do board e diário da equipe preenchidos com informações reais.

### Rubrica — Tarefa 1 (0 a 4,0)

| Critério | Peso | Nota | Observações |
|---|---:|---:|---|
| Problema e unidade de análise | 0,5 | | |
| Coleta bruta e preservação | 1,5 | | |
| Períodos A/B sem rótulo | 1,0 | | |
| Scrum e diário | 1,0 | | |
| **Total** | **4,0** | | |
